# Day 18: The Agent Loop

Yesterday the LLM picked one tool. Today it picks **multiple tools in sequence**.

This is the agent loop: Think → Act → Observe → Repeat.

## The Problem

"What's the cheapest destination and what's the weather there?"

This needs **two** tool calls:
1. Get all prices → find the cheapest
2. Get weather for that city

The agent loop handles this automatically.

## Setup

In [27]:
from google import genai
from google.genai import types
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path='../.env')
API_KEY = os.environ["GEMINI_API_KEY"]
client = genai.Client(api_key=API_KEY)

## Our Tools (Same as Day 17)

In [28]:
# Tool 1: Ticket prices
ticket_prices = {
    "london": 799,
    "paris": 899,
    "tokyo": 1400,
    "berlin": 499,
    "new york": 599
}

def get_ticket_price(destination_city: str) -> str:
    print(f"  🎫 get_ticket_price({destination_city})")
    city = destination_city.lower()
    price = ticket_prices.get(city, None)
    if price:
        return f"Flight to {destination_city}: ${price}"
    return f"No flights to {destination_city}"

def get_all_prices() -> str:
    print(f"  📋 get_all_prices()")
    prices = [f"{city.title()}: ${price}" for city, price in ticket_prices.items()]
    return "Available flights: " + ", ".join(prices)

# Tool 2: Weather
weather_data = {
    "london": {"temp": 15, "condition": "Cloudy"},
    "paris": {"temp": 18, "condition": "Sunny"},
    "tokyo": {"temp": 22, "condition": "Clear"},
    "berlin": {"temp": 12, "condition": "Rainy"},
    "new york": {"temp": 20, "condition": "Partly Cloudy"}
}

def get_weather(city: str) -> str:
    print(f"  🌤️ get_weather({city})")
    weather = weather_data.get(city.lower(), None)
    if weather:
        return f"Weather in {city}: {weather['temp']}°C, {weather['condition']}"
    return f"No weather data for {city}"

# Tool 3: Book flight
def book_flight(destination_city: str, passenger_name: str) -> str:
    print(f"  ✈️ book_flight({destination_city}, {passenger_name})")
    city = destination_city.lower()
    price = ticket_prices.get(city, None)
    if price:
        confirmation = f"BK{hash(passenger_name + city) % 10000:04d}"
        return f"✅ Booked! {passenger_name} → {destination_city}. Confirmation: {confirmation}. Total: ${price}"
    return f"Cannot book to {destination_city}"

## Tool Schemas

In [29]:
travel_tools = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="get_ticket_price",
            description="Get the price of a flight to a specific city.",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "destination_city": types.Schema(type=types.Type.STRING, description="City to fly to")
                },
                required=["destination_city"]
            )
        ),
        types.FunctionDeclaration(
            name="get_all_prices",
            description="Get prices for all available flight destinations.",
            parameters=types.Schema(type=types.Type.OBJECT, properties={})
        ),
        types.FunctionDeclaration(
            name="get_weather",
            description="Get the current weather for a city.",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "city": types.Schema(type=types.Type.STRING, description="City to check weather")
                },
                required=["city"]
            )
        ),
        types.FunctionDeclaration(
            name="book_flight",
            description="Book a flight for a passenger.",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "destination_city": types.Schema(type=types.Type.STRING, description="City to fly to"),
                    "passenger_name": types.Schema(type=types.Type.STRING, description="Passenger name")
                },
                required=["destination_city", "passenger_name"]
            )
        )
    ]
)

print("✅ 4 tools defined")

✅ 4 tools defined


## Tool Dispatcher

In [30]:
def execute_tool(func_call):
    """Route to the correct tool."""
    name = func_call.name
    args = func_call.args
    
    if name == "get_ticket_price":
        return get_ticket_price(args["destination_city"])
    elif name == "get_all_prices":
        return get_all_prices()
    elif name == "get_weather":
        return get_weather(args["city"])
    elif name == "book_flight":
        return book_flight(args["destination_city"], args["passenger_name"])
    else:
        return f"Unknown tool: {name}"

## The Agent Loop

This is the core pattern:
1. Send message to LLM
2. If LLM wants to call a tool → execute it → add result to conversation → loop
3. If LLM gives text response → we're done

In [31]:
def agent(message, max_iterations=5):
    """Agent loop that handles multiple tool calls."""
    print(f"\n🧑 User: {message}")
    print("=" * 50)
    
    # Start conversation
    contents = [types.Content(role="user", parts=[types.Part.from_text(text=message)])]
    
    for i in range(max_iterations):
        print(f"\n🔄 Iteration {i+1}")
        
        # Call LLM
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=contents,
            config=types.GenerateContentConfig(tools=[travel_tools])
        )
        
        part = response.candidates[0].content.parts[0]
        
        # Check if LLM wants to call a tool
        if part.function_call:
            func_call = part.function_call
            print(f"  🔧 Tool: {func_call.name}")
            
            # Execute tool
            result = execute_tool(func_call)
            print(f"  📋 Result: {result}")
            
            # Add tool call and result to conversation
            contents.append(types.Content(role="model", parts=[part]))
            contents.append(types.Content(role="user", parts=[
                types.Part.from_function_response(
                    name=func_call.name,
                    response={"result": result}
                )
            ]))
            
            # Continue loop
            continue
        
        # LLM gave a text response - we're done
        print(f"\n✅ Final Answer:")
        print(response.text)
        return response.text
    
    return "Max iterations reached"

## Test: Single Tool Call

In [32]:
# Simple question - one tool call
agent("How much is a flight to Tokyo?")


🧑 User: How much is a flight to Tokyo?

🔄 Iteration 1
  🔧 Tool: get_ticket_price
  🎫 get_ticket_price(Tokyo)
  📋 Result: Flight to Tokyo: $1400

🔄 Iteration 2

✅ Final Answer:
A flight to Tokyo costs $1400.


'A flight to Tokyo costs $1400.'

## Test: Multiple Tool Calls in Sequence

In [33]:
# Complex question - needs multiple tools
agent("What's the cheapest destination and what's the weather there?")


🧑 User: What's the cheapest destination and what's the weather there?

🔄 Iteration 1
  🔧 Tool: get_all_prices
  📋 get_all_prices()
  📋 Result: Available flights: London: $799, Paris: $899, Tokyo: $1400, Berlin: $499, New York: $599

🔄 Iteration 2
  🔧 Tool: get_weather
  🌤️ get_weather(Berlin)
  📋 Result: Weather in Berlin: 12°C, Rainy

🔄 Iteration 3

✅ Final Answer:
The cheapest destination is Berlin at $499. The weather in Berlin is 12°C, Rainy.


'The cheapest destination is Berlin at $499. The weather in Berlin is 12°C, Rainy.'

In [ ]:
# Another multi-step task
agent("Find me a sunny destination that costs less than $700")

In [ ]:
# Planning + action
agent("Book the cheapest flight for John Doe and tell me what to pack for the weather")

## Key Takeaways

1. **The Loop** — Keep calling tools until the LLM gives a text response
2. **Conversation History** — Each tool result is added to the conversation
3. **Autonomous Reasoning** — The LLM decides what to do next
4. **Max Iterations** — Always have a safety limit

This is the foundation of AI agents. ReAct, function calling agents, and most AI assistants use this pattern.

---

**Next:** Day 19 — Building a complete AI Agent project